# 04 — Options market data: what the options market expected

From 2026Q3 an event may carry a third disclosure item, the **option-implied statistics**: three numbers read off the stock's listed options just before the release. They say how large a move the options market priced, not which way. The item is `id="option-implied-stats"`, `kind="stats"`, `source="option_market"`, with `content` an **object**. It is optional: fewer than half of events have options liquid enough to measure, and no event before 2026Q3 carries it.

This notebook reads the item, then uses it as an **event-specific scale** for a prediction. The score ignores any uniform rescaling of your predictions, so only something that varies across events can move it. We test that on the two reference baselines with the exact scoring transform.

1. What you receive — coverage in the 2026Q3 archive
2. The three numbers, on six events
3. Magnitude, not direction
4. Why the score needs an event-specific transform
5. Two rescalings: naive and adjusted
6. The regression table
7. The competition's own metric
8. Caveats

With a live API key it reads the real 2026Q3 archive; without one, the six real archive lines in `data/sample/`, which carry the item. Sections 3 and 5–7 need the whole quarter and skip in sample mode.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from examples import Client, load_config
from examples.archive import download_archive, read_jsonl_gz
from examples.option_stats import (
    STATISTICS,
    option_implied_stats_from_disclosure,
    option_stats_frame,
    scale_by_implied_move,
    statistic_value,
)
from examples.scoring import (
    BASELINE_LABELS,
    add_percentiles,
    outcomes_frame,
    percentile_ranks,
    score_submission,
)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "sample").is_dir())
ARCHIVE_DIR = REPO / "data" / "archive"  # gitignored download cache
SAMPLE_DIR = REPO / "data" / "sample"

QUARTER = "2026Q3"  # the first quarter with option-implied statistics
TICKERS = ["NVDA", "WMT", "HD", "CRM", "AVGO", "ORCL"]  # notebook 03's six, also bundled
MOVE = "implied_absolute_earnings_move"
REFRESH = False  # True re-downloads the quarter even when a copy is cached (section 1)
MIN_EVENTS = 100  # fewest covered events worth regressing on

config = load_config()
print("Mode:", "LIVE" if config.is_live else "SAMPLE (no EM_API_KEY — using bundled data)")

## 1. What you receive

`option_stats_frame` gives one row per event: the three statistics as float columns (`NaN` when unavailable), `as_of`, `methodology`, and `skew_status`, which says why the skew is missing when the volatility is present.

The live branch downloads only when the quarter is not cached or `REFRESH = True`. The quarter is unsealed, so a cached copy goes stale; a covered count of zero means your copy predates the item.

In [ ]:
path = ARCHIVE_DIR / f"EARNINGS_RELEASE_{QUARTER}.jsonl.gz"
if not config.is_live:
    path = SAMPLE_DIR / f"archive_EARNINGS_RELEASE_{QUARTER}.jsonl.gz"
    print(f"Sample mode: reading {path.name}")
elif REFRESH or not path.exists():
    with Client.from_env() as client:
        manifest = client.archive_manifest()
        (path,) = download_archive(
            manifest,
            ARCHIVE_DIR,
            client=client,
            only=lambda f: f.event_type == "EARNINGS_RELEASE" and f.quarter == QUARTER,
        )
else:
    print(f"Using the cached {path.name}; set REFRESH = True to re-download")

records = list(read_jsonl_gz(path))
stats = option_stats_frame(records)
covered = stats[stats["has_option_stats"]]
n_covered = len(covered)

print(f"{len(records):,} events in {QUARTER}; {n_covered:,} carry option-implied statistics")
if n_covered:
    print(f"  share covered       : {n_covered / len(records):.0%}")
    print(f"  methodology         : {sorted(covered['methodology'].dropna().unique())}")
    print(f"  skew available      : {covered['skew_25_delta'].notna().mean():.0%} of covered")
    print(f"  as_of before cutoff : {bool((covered['as_of'] < covered['knowledge_cutoff']).all())}")
else:
    print("  (none: this copy of the archive predates the item's dissemination)")

In [ ]:
outcomes = outcomes_frame(records)  # one row per (event, asset) that has a car1
models = list(BASELINE_LABELS.values())  # ["Gemini 2.5 Flash-Lite", "GPT-5 nano"]

have_scoring_data = (
    n_covered >= MIN_EVENTS
    and not outcomes.empty
    and set(models) <= set(outcomes.columns)
    and int(outcomes["surprise"].notna().sum()) >= MIN_EVENTS
)
if have_scoring_data:
    print(
        f"Regression sections on: {len(outcomes):,} scored outcomes, {n_covered:,} covered events"
    )
elif not config.is_live:
    print("Sample mode: sections 3 and 5 to 7 need the live archive. Skipping.")
else:
    print(
        f"Live archive, but not enough to regress on ({n_covered:,} covered events; baseline "
        f"predictions {'present' if set(models) <= set(outcomes.columns) else 'absent'}). "
        "Sections 3 and 5 to 7 skip."
    )

## 2. The three numbers, on six events

Each statistic is a `{value, status}` pair; `value` is a decimal (`0.0676` is 6.76%) and `null` unless `status` is `"ok"`.

- `implied_earnings_volatility` — the risk-neutral standard deviation of the jump on the release, backed out of at-the-money implied volatility on the first two expirations after it (Dubinsky, Johannes, Kaeck and Seeger, *RFS* 2019).
- `implied_absolute_earnings_move` — the risk-neutral expected absolute return from that jump, `4·Φ(J/2) − 2 ≈ 0.8·J`. Same information, in return units.
- `skew_25_delta` — 25-delta call minus 25-delta put implied volatility on the nearest expiration. Negative means puts are richer than calls. Missing on its own when the wings are thin.

`as_of` always precedes `knowledge_cutoff`, so the item is a legitimate ex ante input. Below: the raw object for one of notebook 03's six events, then all six.

In [ ]:
def latest_record(ticker: str) -> dict | None:
    hits = [r for r in records if r["focal_assets"][0]["identifier_value"] == ticker]
    return max(hits, key=lambda r: r["event_datetime"]) if hits else None


chosen = {t: r for t in TICKERS if (r := latest_record(t)) is not None}
if not chosen:  # a different quarter, say: fall back to the first six covered events
    ids = set(covered["event_id"].head(6))
    chosen = {r["focal_assets"][0]["identifier_value"]: r for r in records if r["event_id"] in ids}

first = next(iter(chosen.values()))
raw = option_implied_stats_from_disclosure(first)
print(f"{first['event_id']} — the item's content object:")
print(json.dumps(raw, indent=2))
print()
for name in STATISTICS:
    print(f"{name:32s} {statistic_value(raw, name)}")  # None unless status is 'ok'

In [ ]:
six = option_stats_frame(chosen.values()).set_index("tickers").loc[list(chosen)]
six["move / vol"] = six[MOVE] / six["implied_earnings_volatility"]
display(
    six[["as_of", "knowledge_cutoff", *STATISTICS, "skew_status", "move / vol"]].style.format(
        {
            "as_of": "{:%Y-%m-%d %H:%M}",
            "knowledge_cutoff": "{:%Y-%m-%d %H:%M}",
            "implied_earnings_volatility": "{:.2%}",
            MOVE: "{:.2%}",
            "skew_25_delta": "{:+.2%}",
            "move / vol": "{:.3f}",
        },
        na_rep="\u2014",
    )
)

## 3. Magnitude, not direction

The move is about the size of the reaction, not its sign, and it is risk-neutral rather than an unbiased forecast. The skew is an asymmetry in option prices, not a predicted return. Over the quarter the move correlates strongly with the *absolute* realized abnormal return; the skew's correlation with the *signed* return is nil.

In [ ]:
joined = outcomes.merge(stats[["event_id", *STATISTICS]], on="event_id", how="left")

six_ids = {r["event_id"] for r in chosen.values()}
display(
    joined[joined["event_id"].isin(six_ids)]
    .set_index("identifier_value")[["car1", MOVE, "skew_25_delta"]]
    .reindex(list(chosen))  # a very recent live event may have no car1 yet
    .rename(columns={MOVE: "implied move", "skew_25_delta": "skew"})
    .style.format({"car1": "{:+.2%}", "implied move": "{:.2%}", "skew": "{:+.2%}"}, na_rep="\u2014")
)

if have_scoring_data:
    cov = joined.dropna(subset=["car1", MOVE])
    with_skew = cov.dropna(subset=["skew_25_delta"])
    print(f"Covered outcomes: {len(cov):,}")
    print(f"  corr(|car1|, implied move) = {cov['car1'].abs().corr(cov[MOVE]):+.2f}")
    print(f"  corr(car1, implied move)   = {cov['car1'].corr(cov[MOVE]):+.2f}")
    print(
        f"  corr(car1, skew)           = {with_skew['car1'].corr(with_skew['skew_25_delta']):+.2f}"
    )
    print(
        f"  |car1| above implied move  : {(cov['car1'].abs() > cov[MOVE]).mean():.0%} of outcomes"
    )

    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.scatter(cov[MOVE] * 100, cov["car1"].abs() * 100, s=9, alpha=0.45, color="#0072B2", lw=0)
    lim = cov[MOVE].max() * 100
    ax.plot([0, lim], [0, lim], color="#9AA0A6", lw=1, ls="--")  # realized == implied
    ax.set_xlabel("implied absolute earnings move (%)")
    ax.set_ylabel("|car1| realized (%)")
    ax.set_title(f"{QUARTER}: the priced move vs. the realized one")
    ax.grid(color="#D7DBE0", lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    plt.show()

## 4. Why the score needs an event-specific transform

The competition fits `y = α + β₁·ĝ + β₂·s` within a period (`y` the percentile rank of `car1`, `s` that of the surprise, `ĝ` your predicted percentile) and credits the R² gained over the surprise-only fit. That gain is `β₁²·s₁₁.₂ / s_yy` (see `examples.scoring`), so replacing every `p` with `a + b·p` changes nothing: `β₁` absorbs `b`, `α` absorbs `a`. Neither the direction nor the spread of your numbers is scored.

Only a factor that differs across events can move the score. The implied move is one, known before the cutoff. The question is what to multiply, and by how much.

## 5. Two rescalings: naive and adjusted

A prediction is a centre plus a deviation. A stock priced for a 3% move cannot reach the tails of the cross-section, so its deviation should count for less; one priced for 12% can. Ranks are bounded, so the effect saturates: `E[y | ĝ, m] ≈ centre + (ĝ − centre)·g(m)`, `g` increasing and concave.

The **naive** rescale, `0.5 + (ĝ − 0.5)·m/m̄`, gets both wrong. The baselines centre near 0.7, not 0.5, so pivoting at 0.5 pushes every low-move event down and every high-move event up whatever the model said. And a linear factor of up to three or four clips about one event in seven at 0 or 1.

The **adjusted** rescale pivots at the submission's own mean and takes a square root: `μ + (ĝ − μ)·√(m/m̄)`. These are the defaults of `scale_by_implied_move`; the naive rule is `pivot=0.5, power=1.0`. Events without the item keep their prediction. The fit sample is the quarter's covered outcomes with a surprise and both baselines, ranked over the **whole** quarter as the scorer does. First the mechanism: the slope of `y` on `ĝ` by tercile of the implied move.

In [ ]:
if have_scoring_data:
    # Rank over the WHOLE quarter first (the scorer's percentiles), then restrict.
    scored = add_percentiles(outcomes).merge(
        stats[["event_id", *STATISTICS]], on="event_id", how="left"
    )
    fit_df = scored.dropna(subset=["y", "surprise_pct", *models, MOVE]).copy()

    def ranked(values: pd.Series) -> pd.Series:
        return pd.Series(percentile_ranks(values.tolist()), index=values.index)

    fit_df["move_pct"] = ranked(fit_df[MOVE])
    n_no_skew = int(fit_df["skew_25_delta"].isna().sum())
    fit_df["skew_pct"] = ranked(fit_df["skew_25_delta"].fillna(fit_df["skew_25_delta"].mean()))

    for m in models:
        fit_df[f"{m} naive"] = scale_by_implied_move(fit_df[m], fit_df[MOVE], pivot=0.5, power=1.0)
        fit_df[f"{m} adjusted"] = scale_by_implied_move(fit_df[m], fit_df[MOVE])

    def share_clipped(column: str) -> float:
        return float(((fit_df[column] <= 0) | (fit_df[column] >= 1)).mean())

    print(f"Fit sample: N = {len(fit_df):,} covered outcomes with a surprise and both baselines")
    print(
        f"  mean implied move = {fit_df[MOVE].mean():.2%}; skew mean-filled on {n_no_skew:,} rows"
    )
    for m in models:
        print(
            f"  {m}: mean ĝ = {fit_df[m].mean():.3f} over {fit_df[m].nunique()} distinct values; "
            f"clipped {share_clipped(f'{m} naive'):.1%} naive, "
            f"{share_clipped(f'{m} adjusted'):.1%} adjusted"
        )

In [ ]:
if have_scoring_data:
    import statsmodels.api as sm

    fit_df["move tercile"] = pd.qcut(fit_df[MOVE], 3, labels=["low", "mid", "high"])
    rows = []
    for tercile, group in fit_df.groupby("move tercile", observed=True):
        row = {"tercile": tercile, "n": len(group), "mean move": group[MOVE].mean()}
        row["sd(y)"] = group["y"].std()
        for m in models:
            X = sm.add_constant(group[["surprise_pct", m]])
            row[f"slope on ĝ — {m}"] = sm.OLS(group["y"], X).fit().params[m]
        rows.append(row)
    display(
        pd.DataFrame(rows)
        .set_index("tercile")
        .style.format({"mean move": "{:.1%}", "sd(y)": "{:.3f}"}, precision=3)
    )

## 6. The regression table

Eight columns on one fit sample: the surprise alone; the surprise plus the two option statistics as plain regressors (percentile-ranked, skew mean-filled); then for each baseline, the surprise with `ĝ` as submitted, naively scaled, and adjusted. Read the R² row.

As of early October 2026: the option statistics add about 0.003 as regressors; the naive rescale costs Gemini about 0.007 and GPT-5 nano about 0.015; the adjusted rescale adds about 0.005 for Gemini and nothing for GPT-5 nano. Modest, but in the right direction, and a freely fitted interaction does no better. The numbers will drift as the quarter fills in.

In [ ]:
if have_scoring_data:
    import statsmodels.api as sm
    from stargazer.stargazer import Stargazer

    gemini, gpt = models  # display names, in BASELINE_LABELS order
    S, RM, RK = "s (surprise pctile)", "implied move (pctile)", "skew (pctile)"
    G, GN, GA = "ĝ (model pctile)", "ĝ naively scaled", "ĝ adjusted"

    def fit(cols):
        # cols: {display_name -> source column}; shared names let stargazer stack rows.
        X = sm.add_constant(pd.DataFrame({name: fit_df[src] for name, src in cols.items()}))
        return sm.OLS(fit_df["y"].to_numpy(), X).fit()

    fits = [
        fit({S: "surprise_pct"}),  # (1) surprise only
        fit({S: "surprise_pct", RM: "move_pct", RK: "skew_pct"}),  # (2) surprise + options
    ]
    for m in (gemini, gpt):
        fits += [
            fit({S: "surprise_pct", G: m}),  # surprise + model
            fit({S: "surprise_pct", GN: f"{m} naive"}),  # surprise + model, naively scaled
            fit({S: "surprise_pct", GA: f"{m} adjusted"}),  # surprise + model, adjusted
        ]

    star = Stargazer(fits)
    star.title(f"Rescaling the reference baselines by the option-implied move — {QUARTER}")
    star.custom_columns(["Benchmarks", gemini, gpt], [2, 3, 3])
    star.covariate_order(["const", S, RM, RK, G, GN, GA])
    star.rename_covariates({"const": "Intercept"})
    star.dependent_variable_name("r_abn  (percentiled CAR1)")
    star.add_line(
        "ĝ transform",
        ["", "", "as submitted", "naive", "adjusted", "as submitted", "naive", "adjusted"],
    )
    star.show_model_numbers(True)
    display(star)

## 7. The competition's own metric

Section 6 used a common fit sample. Here each transform is applied as you would submit it, over every scored outcome in the quarter with uncovered events keeping `ĝ`, and `score_submission` reports `delta_r_squared`.

In [ ]:
if have_scoring_data:
    full = scored.copy()  # every scored outcome in the quarter; NaN move -> ĝ unchanged
    rows = []
    for m in models:
        full[f"{m} naive"] = scale_by_implied_move(full[m], full[MOVE], pivot=0.5, power=1.0)
        full[f"{m} adjusted"] = scale_by_implied_move(full[m], full[MOVE])
        for label, column in (
            ("as submitted", m),
            ("naive", f"{m} naive"),
            ("adjusted", f"{m} adjusted"),
        ):
            result = score_submission(full, column)
            rows.append(
                {
                    "baseline": m,
                    "ĝ transform": label,
                    "n_obs": result["n_obs"],
                    "R² surprise": result["r_squared_surprise"],
                    "R²": result["r_squared"],
                    "ΔR²": result["delta_r_squared"],
                }
            )
    display(
        pd.DataFrame(rows)
        .set_index(["baseline", "ĝ transform"])
        .style.format({"R² surprise": "{:.4f}", "R²": "{:.4f}", "ΔR²": "{:+.4f}"})
    )

## 8. Caveats

- The move is a **risk-neutral magnitude**: no direction, and the realized absolute return exceeds it more than a third of the time.
- **Coverage is partial.** Fewer than half of events carry the item, mostly larger companies, so most predictions are untouched; hence section 7's gain is smaller than section 6's.
- **One quarter, in sample.** A few thousandths of R², with the pivot and reference scale computed on the same data. The lesson is the shape of the result, not its size.
- A multi-asset event carries one item, attached above to each focal asset.

## What to take away

- The item is optional. Match on `id` and `kind`, read each `status`, and build for its absence.
- The move is about `0.8` times the volatility, so use one of the two. The skew is not a direction.
- The score ignores any uniform remap of your predictions; a per-event input helps only through how it differs across events.
- Pivot at your own centre and let the factor saturate. `scale_by_implied_move` does both by default; `pivot=0.5, power=1.0` shows what the naive version costs.

## Where to go from here

The item reaches a live event before its `knowledge_cutoff`, so a deployed handler can apply `scale_by_implied_move` to its own prediction before `POST /predictions`, with its typical prediction as the pivot and a running mean of past moves as the scale. For the scoring transform see section 6 of [`01_historical_archive.ipynb`](01_historical_archive.ipynb); for the preview beside this item, [`03_earnings_previews.ipynb`](03_earnings_previews.ipynb).